# Coin Toss

Flip a quantum coin a number of times and get back the heads and tails counts.

## Setup

In [1]:
import getpass
import os

from moth_atlas import Atlas

ENGINE = "coin-toss-v1"  # the engine this notebook runs

if not os.environ.get("MOTH_API_KEY"):
    os.environ["MOTH_API_KEY"] = getpass.getpass("Atlas API key")

client = Atlas()  # reads MOTH_API_KEY

Atlas API key ········


## Try it

Set the parameters, then run the cell.

In [3]:
job = client.jobs.run(
    engine=ENGINE,
    params={
        "shots": 10,
    },
)
job.result

coin-toss-v1 job c7e10f11 submitted
coin-toss-v1 job c7e10f11 completed (2s)


{'backend': 'aer',
 'heads': 6,
 'ibm_job_id': 'd11f78f0abad418a9ef74c9697cc3c76',
 'io': {'backend': None,
  'dynamical_decoupling': None,
  'emu_trajectories': None,
  'error_mitigation': None,
  'initial_layout': None,
  'layout': None,
  'mode': 'sim',
  'optimization_level': None,
  'precision': None,
  'provider': None,
  'routing': None,
  'seed': None,
  'seed_sampler': None,
  'seed_transpiler': None,
  'shots': 10,
  'sim_noise': None,
  'store': True,
  'time_budget': False,
  'transpile_mode': None,
  'twirling': None},
 'io_report': {'applied': ['backend=None',
   'mode=sim',
   'provider=None',
   'shots=10',
   'store=True',
   'time_budget=False'],
  'ignored': [],
  'unsupported': []},
 'mode': 'emu',
 'output': 'heads',
 'shots': 10,
 'tails': 4}

## Parameters

What to expect from each:

- `shots` (above 0): more shots bring the heads/tails split closer to 50/50.
- `mode`: `emu` runs on the platform's simulator and returns in seconds; `qpu` runs on real IBM Quantum hardware and can queue for minutes.
- `backend_name`: the IBM Quantum backend for `qpu`; leave it out to let the platform choose the least busy one.

It returns `heads` and `tails` (counts), `shots` and `output` (whichever had more counts).

In [ ]:
# client.engines.get(ENGINE)  # its description, params, input and output files, from the API

## How it works

A single qubit is put into equal superposition with a Hadamard gate and measured `shots` times. The outcomes are counted and the majority is reported as the result.

## Experiments

Each experiment changes one parameter and keeps the others fixed, and runs one job per value.

### Shots

More shots bring the heads/tails split closer to 50/50.

In [5]:
for shots in (10, 100, 1000):
    print(f"shots={shots}")
    job = client.jobs.run(engine=ENGINE, params={"shots": shots}, quiet=True)
    r = job.result
    print(f"  {r['heads']} heads, {r['tails']} tails ({r['heads'] / shots:.0%} heads)")

shots=10
  5 heads, 5 tails (50% heads)
shots=100
  52 heads, 48 tails (52% heads)
shots=1000
  513 heads, 487 tails (51% heads)


## Questions

**Is this a random-number service?**

No. The circuit is one qubit; results are a probability sample, not a random-number service with guarantees.

**Why did my qpu run fail?**

A run still going after 300 s is cancelled and the job ends failed. Device queue times can be minutes to hours.